In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Core imports
import os, json, random, math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2025-12-24 16:24:17.926313: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766593458.391055      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766593458.505873      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766593459.613314      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766593459.613353      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766593459.613356      55 computation_placer.cc:177] computation placer alr

/kaggle/input/data4good-test/test.json
/kaggle/input/data4good/train.json


In [2]:
import gc, torch

# del trainer, model
gc.collect()
torch.cuda.empty_cache()

In [3]:
# -----------------------
# Config
# -----------------------
MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
LABELS = ["factual", "contradiction", "irrelevant"]

TRAIN_PATH = "/kaggle/input/data4good/train.json"
TEST_PATH  = "/kaggle/input/data4good-test/test.json"

OUT_DIR = "outputs_kfold"
os.makedirs(OUT_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

MAX_LENGTH = 256
LR = 2e-5
EPOCHS = 2
TRAIN_BATCH = 8
EVAL_BATCH = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [5]:
def load_json_records(path: str) -> List[Dict]:
    # supports list-of-dicts, or dict-wrapped list-of-dicts
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                return v
    raise ValueError(f"Unsupported JSON format in {path}")

def normalize_label(x) -> str:
    return str(x).strip().lower()

def build_label_maps(labels: List[str]) -> Tuple[Dict[str, int], Dict[int, str]]:
    labels_norm = [normalize_label(l) for l in labels]
    label2id = {l:i for i,l in enumerate(labels_norm)}
    id2label = {i:l for l,i in label2id.items()}
    return label2id, id2label

label2id, id2label = build_label_maps(LABELS)
label2id, id2label


({'factual': 0, 'contradiction': 1, 'irrelevant': 2},
 {0: 'factual', 1: 'contradiction', 2: 'irrelevant'})

In [6]:
def prepare_dataframe(records: List[Dict], is_train: bool) -> pd.DataFrame:
    df = pd.DataFrame(records)

    # expected keys (most common in this competition)
    # - question, context, answer, type
    # but we make this robust to variations.
    if "answer" not in df.columns:
        raise ValueError(f"Missing 'answer' column. Found columns: {list(df.columns)}")

    # Build premise = context + question
    if "context" in df.columns:
        ctx = df["context"].fillna("").astype(str)
    elif "premise" in df.columns:
        ctx = df["premise"].fillna("").astype(str)
    else:
        raise ValueError(f"Missing 'context' (or 'premise') column. Found columns: {list(df.columns)}")

    if "question" in df.columns:
        q = df["question"].fillna("").astype(str)
    else:
        # fallback if question key is different
        # common alternates: 'query', 'prompt'
        for alt in ["query", "prompt"]:
            if alt in df.columns:
                q = df[alt].fillna("").astype(str)
                break
        else:
            # if no question exists, just use context as premise
            q = pd.Series([""] * len(df))

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = df["answer"].fillna("").astype(str)

    if is_train:
        if "type" not in df.columns:
            raise ValueError(f"Missing 'type' label column. Found columns: {list(df.columns)}")
        df["label"] = df["type"].apply(normalize_label).map(label2id)
        if df["label"].isna().any():
            bad = df[df["label"].isna()]["type"].unique()[:10]
            raise ValueError(f"Found unknown labels in train: {bad}. Expected {LABELS}")
        df["label"] = df["label"].astype(int)

    return df

train_records = load_json_records(TRAIN_PATH)
test_records  = load_json_records(TEST_PATH)

train_df = prepare_dataframe(train_records, is_train=True)
test_df  = prepare_dataframe(test_records,  is_train=False)

train_df.head(2)


,answer,type,context,question,premise_text,hypothesis_text,label
0,"In 1512, Parliament passed a significant act t...",factual,During the Hundred Years' War a French attack ...,In what year did Parliament pass a notable law...,During the Hundred Years' War a French attack ...,"In 1512, Parliament passed a significant act t...",0
1,The Spanish and French were the ones who estab...,factual,"""By May 1539, Conquistador Hernando de Soto sk...",Who established early settlements in Florida,"""By May 1539, Conquistador Hernando de Soto sk...",The Spanish and French were the ones who estab...,0


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(df: pd.DataFrame, with_labels: bool) -> Dataset:
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    remove_cols = [c for c in ds.column_names if c not in ["premise_text","hypothesis_text","label"]]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)
    if with_labels:
        ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids","attention_mask"] + (["labels"] if with_labels else []))
    return ds


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [8]:
def make_model():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, 
                                                               num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,)

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        # older transformers: no kwargs support, fallback
        model.gradient_checkpointing_enable()

    model.config.use_cache = False 

    return model


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))
    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "confusion_matrix": cm.tolist(),
    }

import inspect
from transformers import TrainingArguments

def trainer_args(run_name: str, out_dir: str) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)
    allowed = set(sig.parameters.keys())

    # Map arg name changes across transformers versions
    if "eval_strategy" in allowed and "evaluation_strategy" not in allowed:
        eval_key = "eval_strategy"
    else:
        eval_key = "evaluation_strategy"

    kwargs = dict(
        output_dir=out_dir,
        run_name=run_name,
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        warmup_ratio=0.06,
        fp16=True,

        # evaluation/checkpointing/logging
        **{eval_key: "epoch"},
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,

        report_to="none",
        seed=42,
    )

    # Drop anything unsupported by this transformers version
    kwargs = {k: v for k, v in kwargs.items() if k in allowed}

    # If the version lacks run_name, drop it
    # (filter above already does this, but keeping comment for clarity)
    return TrainingArguments(**kwargs)


def predict_probs(trainer: Trainer, ds: Dataset, batch_size: int = 64):
    # Trainer.predict uses the trainer's args eval batch size, but we can override with a temp args if needed.
    preds = trainer.predict(ds)
    logits = preds.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=-1)
    conf = probs.max(axis=-1)
    return pred_ids, conf, probs


## 5 fold Cross validation

In [9]:
# skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# oof_pred = np.zeros(len(train_df), dtype=int)
# oof_prob = np.zeros((len(train_df), len(LABELS)), dtype=float)

# fold_metrics = []

# for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
#     print(f"\n===== Fold {fold+1}/{N_SPLITS} =====")
#     fold_dir = os.path.join(OUT_DIR, f"fold_{fold}")
#     os.makedirs(fold_dir, exist_ok=True)

#     tr_df = train_df.iloc[tr_idx].copy()
#     va_df = train_df.iloc[va_idx].copy()

#     tr_ds = tokenize_dataset(tr_df, with_labels=True)
#     va_ds = tokenize_dataset(va_df, with_labels=True)

#     model = make_model()
#     args = trainer_args(run_name=f"fold_{fold}", out_dir=fold_dir)

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=tr_ds,
#         eval_dataset=va_ds,
#         tokenizer=tokenizer,
#         compute_metrics=compute_metrics,
#     )

#     trainer.train()

#     # OOF predictions
#     va_pred_ids, va_conf, va_probs = predict_probs(trainer, tokenize_dataset(va_df, with_labels=False))
#     oof_pred[va_idx] = va_pred_ids
#     oof_prob[va_idx] = va_probs

#     # metrics
#     y_true = va_df["label"].values
#     acc = accuracy_score(y_true, va_pred_ids)
#     macro_f1 = f1_score(y_true, va_pred_ids, average="macro", zero_division=0)
#     fold_metrics.append({"fold": fold, "acc": acc, "macro_f1": macro_f1})
#     print(f"Fold acc={acc:.4f}  macro_f1={macro_f1:.4f}")

# cv_df = pd.DataFrame(fold_metrics)
# cv_df


In [10]:
# print("CV mean macro_f1:", cv_df["macro_f1"].mean())
# print("CV std  macro_f1:", cv_df["macro_f1"].std())

# # Save OOF predictions (useful for analysis/blending)
# oof_out = train_df[["premise_text","hypothesis_text","type"]].copy()
# oof_out["true_label"] = train_df["type"].apply(normalize_label)
# oof_out["pred_id"] = oof_pred
# oof_out["pred_label"] = [id2label[i] for i in oof_pred]
# for i,lbl in enumerate(LABELS):
#     oof_out[f"prob_{lbl}"] = oof_prob[:, i]
# oof_path = os.path.join(OUT_DIR, "oof_predictions.csv")
# oof_out.to_csv(oof_path, index=False)
# oof_path


## Train on full data and predict on train + test

In [9]:
full_dir = os.path.join(OUT_DIR, "full_train")
os.makedirs(full_dir, exist_ok=True)

full_train_ds = tokenize_dataset(train_df, with_labels=True)

full_model = make_model()
full_args = TrainingArguments(
    output_dir=full_dir,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

full_trainer = Trainer(
    model=full_model,
    args=full_args,
    train_dataset=full_train_ds,
    tokenizer=tokenizer,
)
full_trainer.train()


Map:   0%|          | 0/21021 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

/tmp/ipykernel_55/1513794661.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  full_trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
50,0.679900
100,0.165500
150,0.125100
200,0.066000
250,0.083100
300,0.088000
350,0.042200
400,0.035400
450,0.033300
500,0.046600


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=2628, training_loss=0.046027474355907996, metrics={'train_runtime': 6671.8009, 'train_samples_per_second': 6.301, 'train_steps_per_second': 0.394, 'total_flos': 1.9349788614182964e+16, 'train_loss': 0.046027474355907996, 'epoch': 2.0})

In [10]:
# Predict on TRAIN (full model)
train_pred_ids, train_conf, train_probs = predict_probs(full_trainer, tokenize_dataset(train_df, with_labels=False))

train_pred_df = train_df.copy()
train_pred_df["pred_id"] = train_pred_ids
train_pred_df["pred_label"] = [id2label[i] for i in train_pred_ids]
train_pred_df["pred_conf"] = train_conf
for i,lbl in enumerate(LABELS):
    train_pred_df[f"prob_{lbl}"] = train_probs[:, i]

train_pred_path = os.path.join(OUT_DIR, "train_predictions_fullmodel.csv")
train_pred_df.to_csv(train_pred_path, index=False)

# Predict on TEST (full model)
test_pred_ids, test_conf, test_probs = predict_probs(full_trainer, tokenize_dataset(test_df, with_labels=False))

test_pred_df = test_df.copy()
test_pred_df["pred_id"] = test_pred_ids
test_pred_df["pred_label"] = [id2label[i] for i in test_pred_ids]
test_pred_df["pred_conf"] = test_conf
for i,lbl in enumerate(LABELS):
    test_pred_df[f"prob_{lbl}"] = test_probs[:, i]

test_pred_path = os.path.join(OUT_DIR, "test_predictions_fullmodel.csv")
test_pred_df.to_csv(test_pred_path, index=False)

(train_pred_path, test_pred_path)


Map:   0%|          | 0/21021 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

('outputs_kfold/train_predictions_fullmodel.csv',
 'outputs_kfold/test_predictions_fullmodel.csv')

In [11]:
# Submission file (if needed): usually expects id + type (label)
# Adjust column name if your competition uses a different ID key.
id_col = "ID"

if id_col is None:
    print("No obvious id column found in test. Columns:", list(test_pred_df.columns))
else:
    sub = test_pred_df[[id_col, "pred_label"]].rename(columns={"pred_label":"type"})
    sub_path = os.path.join(OUT_DIR, "submission.csv")
    sub.to_csv(sub_path, index=False)
    print("Saved:", sub_path)


Saved: outputs_kfold/submission.csv


In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

In [15]:
confusion_matrix(train_pred_df.type, train_pred_df.pred_label)

array([[ 1808,     9,     1],
       [    1, 17429,     1],
       [    1,     0,  1771]])

In [18]:
f1_score(train_pred_df.type, train_pred_df.pred_label, average = 'macro')

0.9985102212337503

In [19]:
accuracy_score(train_pred_df.type, train_pred_df.pred_label)

0.9993815708101422